# Bloch States, Berry Curvature and the Chern Number

**Abstract.** This notebook turns the Bloch Hamiltonian $H(\mathbf k)$ of the previous notebook into a topological invariant. We derive the Berry connection and curvature of a Bloch band, identify the Chern number as the integral of the curvature over the Brillouin zone, and compute it numerically with the package's `Chern_number_Fukui_Hatsugai_Suzuki`. As a worked example we build the Haldane model and map out its topological phase diagram as a Chern-number heatmap.

**References**

- N. W. Ashcroft and N. D. Mermin, *Solid State Physics* (Saunders College Publishing, 1976), Chs. 12–13.
- D. N. Sheng et al., Phys. Rev. Lett. **107**, 146803 (2011).
- K. Sun, Z. Gu, H. Katsura, and S. Das Sarma, Phys. Rev. Lett. **106**, 236803 (2011).
- T. Fukui, Y. Hatsugai, and H. Suzuki, J. Phys. Soc. Jpn. **74**, 1674 (2005).
- R. Resta, Rev. Mod. Phys. **66**, 899 (1994).

> **Execution note.** Authored without `nbconvert`, so cells ship *unexecuted but correct*; the phase-diagram SVG was produced by running the identical code out-of-band with the package venv.


## 1. Bloch states and the Berry connection

For each $\mathbf k$ the periodic-gauge Bloch Hamiltonian has orthonormal eigenstates

\begin{equation}
H(\mathbf k)\,|u_n(\mathbf k)\rangle = \varepsilon_n(\mathbf k)\,|u_n(\mathbf k)\rangle,
\qquad
\langle u_n(\mathbf k)|u_n(\mathbf k)\rangle = 1 .
\end{equation}

A band $n$ has a $\mathrm U(1)$ gauge freedom: $|u_n(\mathbf k)\rangle\to e^{i\chi(\mathbf k)}|u_n(\mathbf k)\rangle$ leaves all physical quantities invariant. Quantities built from $\mathbf k$-derivatives must be gauge-covariant. The **Berry connection** (gauge field)

\begin{equation}
\mathbf A_n(\mathbf k) = i\,\langle u_n(\mathbf k)|\boldsymbol\nabla_{\mathbf k}|u_n(\mathbf k)\rangle
= -\,\mathrm{Im}\,\langle u_n|\boldsymbol\nabla_{\mathbf k}u_n\rangle
\end{equation}

transforms as $\mathbf A_n\to\mathbf A_n-\boldsymbol\nabla_{\mathbf k}\chi$, so its curl is gauge-invariant. That curl is the **Berry curvature**

\begin{equation}
\Omega_n(\mathbf k) = \boldsymbol\nabla_{\mathbf k}\times\mathbf A_n(\mathbf k).
\end{equation}

In two dimensions $\Omega_n$ is a single scalar (the $z$-component). For an isolated band the integral of $\Omega_n$ over the Brillouin zone is quantised to an integer — the **Chern number**:

\begin{equation}
\boxed{\; C_n = \frac{1}{2\pi}\int_{\mathrm{BZ}}\Omega_n(\mathbf k)\,d^2k \in \mathbb Z \;}.
\end{equation}

> **Why integer?** The Berry connection over a closed surface defines a $\mathrm U(1)$ bundle whose first Chern class integrates to an integer (Gauss–Bonnet–Chern). The integer survives any smooth gauge choice and can change only when the gap between band $n$ and its neighbours closes — the hallmark of a topological phase transition.


## 2. The Fukui–Hatsugai–Suzuki lattice gauge

Evaluating $\boldsymbol\nabla_{\mathbf k}|u_n\rangle$ by finite differences is fragile because each eigen-solver picks an arbitrary phase. Fukui, Hatsugai and Suzuki (J. Phys. Soc. Jpn. **74**, 1674 (2005)) avoid derivatives entirely by using **overlap (link) variables** between neighbouring grid points.

Discretise the BZ into an $n_k\times n_k$ mesh with sites $\mathbf k_{ij}=(i/n_k,\,j/n_k)$ and define the link

\begin{equation}
U_\mu(\mathbf k_{ij}) =
\frac{\langle u_n(\mathbf k_{ij})|u_n(\mathbf k_{ij}+\hat\mu)\rangle}
{\bigl|\langle u_n(\mathbf k_{ij})|u_n(\mathbf k_{ij}+\hat\mu)\rangle\bigr|},
\qquad \mu\in\{x,y\}.
\end{equation}

The **lattice field strength** of a plaquette is

\begin{equation}
F_{12}(\mathbf k_{ij}) = \ln\!\Bigl[
U_x(\mathbf k_{ij})\,U_y(\mathbf k_{ij}+\hat x)\,
U_x(\mathbf k_{ij}+\hat y)^{-1}\,U_y(\mathbf k_{ij})^{-1}\Bigr] \in (-\pi,\pi],
\end{equation}

and the Chern number is the sum over all plaquettes,

\begin{equation}
C_n = \frac{1}{2\pi i}\sum_{ij}F_{12}(\mathbf k_{ij}).
\end{equation}

For a sufficiently dense mesh this converges to the exact integer, provided the band is separated by a gap (so $|u_n\rangle$ varies smoothly and no overlap vanishes). The package implements exactly this as `Chern_number_Fukui_Hatsugai_Suzuki(Hk_crys, band=..., nk=...)` with **1-based** `band`.


## 3. Sanity check — a trivial insulator

The gapped graphene model $H(\mathbf k)=\mathbf d(\mathbf k)\cdot\boldsymbol\sigma$ with $\mathbf d=(\mathrm{Re}\,f,\,-\mathrm{Im}\,f,\,M)$ has $C=0$ for any $M\neq 0$ (the gap never closes). We verify this before moving to the topological Haldane model.


In [ ]:
import numpy as np
from tightbinding_py import *

lat = initialize_real_space_lattice(lattice_name="honeycomb", sample_size=[3, 3],
                                    pbc_indicator=[True, True])
tb = initialize_real_space_tightbinding_model(lat, model_name="gapped_graphene")

# nearest neighbour t = -1, staggered mass M = 0.5
add_hopping_term(tb, ((((0, 0), 1), ((0, 0), 2)), -1.0))
add_hopping_term(tb, ((((0, 0), 1), ((0, -1), 2)), -1.0))
add_hopping_term(tb, ((((0, 0), 1), ((-1, 0), 2)), -1.0))
add_hopping_term(tb, ((((0, 0), 1), ((0, 0), 1)), 0.5), is_hermitian=False)
add_hopping_term(tb, ((((0, 0), 2), ((0, 0), 2)), -0.5), is_hermitian=False)

Hk_trivial = build_Hk_crys(tb)
print("trivial insulator  C_band1 =", Chern_number_Fukui_Hatsugai_Suzuki(Hk_trivial, band=1, nk=31))
print("trivial insulator  C_band2 =", Chern_number_Fukui_Hatsugai_Suzuki(Hk_trivial, band=2, nk=31))


## 4. The Haldane model

Haldane's 1988 model is graphene plus two ingredients that, together, break time-reversal symmetry without a net magnetic field:

1. a **staggered sublattice potential** $+M$ on $A1$, $-M$ on $A2$ (breaks inversion);
2. complex **next-nearest-neighbour (NNN) hoppings** $t_2 e^{\pm i\varphi}$ whose sign is set by the chirality of the path (breaks time-reversal).

In the crystal gauge the model is the template list below (the same convention as the package `README` example). Writing $H(\mathbf k)=d_0(\mathbf k)\sigma_0+\mathbf d(\mathbf k)\cdot\boldsymbol\sigma$, the NNN terms give the diagonal parts while the NN terms give the off-diagonal $d_x,d_y$, which vanish at the Dirac points. At the Dirac points $K=(2/3,1/3)$ and $K'=(1/3,2/3)$ the gap is set by $d_z$ alone:

\begin{equation}
\Delta_K = d_z(K) = M + 3\sqrt3\,t_2\sin\varphi,
\qquad
\Delta_{K'} = d_z(K') = M - 3\sqrt3\,t_2\sin\varphi .
\end{equation}

The Chern number is

\begin{equation}
C = \tfrac12\bigl[\operatorname{sgn}(\Delta_K) - \operatorname{sgn}(\Delta_{K'})\bigr],
\end{equation}

so the system is topological whenever the two Dirac masses have opposite signs, i.e. $|M| < 3\sqrt3\,|t_2\sin\varphi|$.


In [ ]:
def build_haldane_model(M, phi, t1=-1.0, t2=-0.24, sample_size=None):
    '''Build the Haldane model via add_hopping_term (1-based sublattices).'''
    lat = initialize_real_space_lattice(
        sample_size=sample_size if sample_size is not None else [3, 3],
        lattice_name="honeycomb", pbc_indicator=[True, True],
    )
    tb = initialize_real_space_tightbinding_model(lat, model_name="haldane")
    # nearest neighbour (t1)
    add_hopping_term(tb, ((((0, 0), 1), ((0, 0), 2)), t1))
    add_hopping_term(tb, ((((0, 0), 1), ((0, -1), 2)), t1))
    add_hopping_term(tb, ((((0, 0), 1), ((-1, 0), 2)), t1))
    # next-nearest neighbour (complex, chirality-dependent)
    for src in [1, 2]:
        sgn = 1 if src == 1 else -1
        add_hopping_term(tb, ((((0, 0), src), ((1, 0), src)), t2 * np.exp(sgn * 1j * phi)))
        add_hopping_term(tb, ((((0, 0), src), ((0, 1), src)), t2 * np.exp(-sgn * 1j * phi)))
        add_hopping_term(tb, ((((0, 0), src), ((-1, 1), src)), t2 * np.exp(sgn * 1j * phi)))
    # staggered sublattice potential (mass)
    add_hopping_term(tb, ((((0, 0), 1), ((0, 0), 1)), M), is_hermitian=False)
    add_hopping_term(tb, ((((0, 0), 2), ((0, 0), 2)), -M), is_hermitian=False)
    return tb


tb_haldane = build_haldane_model(M=0.7, phi=np.pi / 2)
Hk_haldane = build_Hk_crys(tb_haldane)
print("Haldane (M=0.7, t2=-0.24, phi=pi/2):")
print("   C_band1 (lower) =", Chern_number_Fukui_Hatsugai_Suzuki(Hk_haldane, band=1, nk=31))
print("   C_band2 (upper) =", Chern_number_Fukui_Hatsugai_Suzuki(Hk_haldane, band=2, nk=31))


Both bands are quantised: $C_1 = -1$ (lower) and $C_2 = +1$ (upper), summing to zero as required for a two-band model (the total Chern number of all bands must vanish). The lower band is a Chern insulator carrying one unit of Hall conductance $\sigma_{xy} = C e^2/h$.


## 5. The topological phase diagram

We scan the mass $M$ and the NNN phase $\varphi$ and compute $C_1$ on a coarse mesh. The topological lobes (the Haldane "eye") are bounded by $|M| = 3\sqrt3\,|t_2\sin\varphi|$; inside, $C_1 = -\operatorname{sgn}(\varphi)$, and the sign flips when $\varphi$ crosses $0$ or $\pm\pi$ (where $\sin\varphi$ vanishes and the gap closes). Because the Chern number is computed from the *infinite-system* templates of `input_hopping_map`, a small real-space sample suffices.


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

FIG_DIR = Path("doc") / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

Ms = np.linspace(-1.4, 1.4, 31)
phis = np.linspace(-np.pi, np.pi, 31)

C_lower = np.zeros((len(Ms), len(phis)))
for iM, M in enumerate(Ms):
    for iphi, phi in enumerate(phis):
        tb = build_haldane_model(M=M, phi=phi)
        C_lower[iM, iphi] = Chern_number_Fukui_Hatsugai_Suzuki(
            build_Hk_crys(tb), band=1, nk=15)

fig, ax = plt.subplots(figsize=(7.0, 5.2))
im = ax.pcolormesh(phis, Ms, C_lower, shading="nearest",
                   cmap="RdBu_r", vmin=-1.5, vmax=1.5)
ax.contour(phis, Ms, C_lower, levels=[-0.5, 0.5], colors="k", linewidths=0.8)
ax.set_xlabel(r"NNN phase $\varphi$ (rad)")
ax.set_ylabel(r"staggered mass $M$  ($t_1=-1$)")
ax.set_title("Haldane phase diagram: Chern number $C_1$ of the lower band")
cb = fig.colorbar(im, ax=ax)
cb.set_label("Chern number $C_1$")
fig.tight_layout()
fig.savefig(FIG_DIR / "haldane_phase_diagram.svg")

# Spot checks with a finer mesh.
print("C1 at (M=0.2, phi=+1.0):", Chern_number_Fukui_Hatsugai_Suzuki(
    build_Hk_crys(build_haldane_model(M=0.2, phi=1.0)), band=1, nk=31))
print("C1 at (M=0.2, phi=-1.0):", Chern_number_Fukui_Hatsugai_Suzuki(
    build_Hk_crys(build_haldane_model(M=0.2, phi=-1.0)), band=1, nk=31))
print("C1 at (M=1.4, phi=pi/2):", Chern_number_Fukui_Hatsugai_Suzuki(
    build_Hk_crys(build_haldane_model(M=1.4, phi=np.pi/2)), band=1, nk=31))


The heatmap shows the classic Haldane "eye": a blue lobe $C_1=-1$ for $\varphi>0$ and a red lobe $C_1=+1$ for $\varphi<0$, both clipped by the boundary $|M|=3\sqrt3\,|t_2\sin\varphi|$, with $C_1=0$ (trivial) outside. The two lobes meet at the gap-closing lines, where the Chern number jumps by one.


## 6. Finite-size and mesh effects

- **Mesh convergence.** FHS converges rapidly for a gapped band: $n_k=31$ already gives $C$ to machine precision for the Haldane parameters above. The heatmap uses a coarser $n_k=15$ for speed; near a gap-closing transition the eigenstates are not smooth and a finite mesh rounds the jump, so the contour lines slightly blur the true phase boundary (compare the finer $n_k=31$ spot checks).
- **Accidental degeneracies.** If two bands touch inside the BZ (a Dirac or quadratic point), the single-band FHS formula is ill-defined there; one must treat the *entire* occupied subspace (the Slater-determinant generalisation of the next notebook) or add a small symmetry-breaking term to open a gap.
- **Gauge freedom.** The eigenvectors returned by the eigensolver have arbitrary phases, but the link variables are built from *ratios* of overlaps whose common phase cancels, so the final integer is gauge-independent.
- **Real-space size vs k-mesh.** The Chern number is a property of the *infinite* system: here it is computed from $H(\mathbf k)$ built from the translation-invariant templates. The finite real-space sample matters only for the many-body generalisation of the next notebook, where the flux torus plays the role of the Brillouin zone.
